# Zero-Shot Mutation Effect Prediction

**Tier 5 — Modern AI for Science | Module 06 · Notebook 2**

*Prerequisites: Notebook 1 (ESM2 embeddings)*

---

**By the end of this notebook you will be able to:**
1. Compute mutation effect scores from sequence likelihood proxies
2. Rank variants without supervised task labels
3. Compare predicted mutation effects against synthetic truth
4. Build confidence-aware mutation triage tables

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook demonstrates how foundation models are used in modern computational biology for representation learning, prioritization, and hypothesis generation.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Model score != biological truth. Treat predictions as ranked hypotheses requiring validation.
- Context length and tokenization can change model behavior more than minor hyperparameter tweaks.
- Domain shift is common: performance on curated benchmarks may not transfer to your assay/data source.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← ESM2 Embeddings](./01_esm2_embeddings.ipynb) | [Module Overview](README.md)

---

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(13)
AA = list('ACDEFGHIKLMNPQRSTVWY')

## 1. Zero-shot scoring intuition

A common score is relative log-likelihood:

\( \Delta = \log P(\text{mutant}|context) - \log P(\text{wildtype}|context) \)

Negative values suggest lower plausibility under the learned sequence distribution.

In [ ]:
HYDRO = set('AILMFWVY')
POLAR = set('STNQ')
CHARGED = set('KRHDE')

def aa_group(a):
    if a in HYDRO:
        return 'hydro'
    if a in POLAR:
        return 'polar'
    if a in CHARGED:
        return 'charged'
    return 'other'

def toy_zero_shot_delta(wt: str, mut: str) -> float:
    # toy proxy: penalize physicochemical group switches
    return 0.25 if aa_group(wt) == aa_group(mut) else -0.55

## 2. Score all single substitutions in a region

In [ ]:
wt_seq = 'MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ'

records = []
for i, wt in enumerate(wt_seq):
    for mut in AA:
        if mut == wt:
            continue
        delta = toy_zero_shot_delta(wt, mut)
        records.append({'pos': i + 1, 'wt': wt, 'mut': mut, 'delta_ll': delta})

df = pd.DataFrame(records)
df.sort_values('delta_ll').head(8)

## 3. Synthetic benchmark against pseudo-fitness

We generate pseudo-fitness scores correlated with delta values and check rank correlation.

In [ ]:
noise = np.random.normal(0, 0.08, size=len(df))
df['fitness'] = 0.6 + 0.5 * df['delta_ll'] + noise

corr = df[['delta_ll', 'fitness']].corr(method='spearman').iloc[0, 1]
print('Spearman correlation (delta vs fitness):', round(float(corr), 3))

## 4. Clinically oriented triage table

Combine zero-shot effect with rarity and structural confidence proxies.

In [ ]:
subset = df.sample(12, random_state=1).copy()
subset['rarity'] = np.random.uniform(0.2, 1.0, size=len(subset))
subset['plddt_proxy'] = np.random.uniform(55, 95, size=len(subset))

subset['priority'] = (
    0.5 * (-subset['delta_ll']) +
    0.3 * subset['rarity'] +
    0.2 * (subset['plddt_proxy'] / 100.0)
)

subset.sort_values('priority', ascending=False)[['pos', 'wt', 'mut', 'delta_ll', 'rarity', 'plddt_proxy', 'priority']]

## Summary

- Zero-shot mutation scoring is useful when labels are scarce.
- Relative likelihood deltas provide actionable first-pass ranking.
- Integrating sequence score with rarity and structure confidence improves prioritization quality.

## Source-backed Context

- ESM-1v/related models are commonly used for zero-shot variant scoring workflows.
- ProteinGym is a widely used benchmark reference point for mutation-effect evaluation.


## Validated Sources

Checked online during content expansion.

- [ESM repository (ESM-1v/ESM-2 variants)](https://github.com/facebookresearch/esm)
- [ProteinGym benchmark](https://proteingym.org/)
